# TRIBE v2 quick test\nThis notebook runs Meta's pretrained TRIBE v2 model on a short video and saves the predicted cortical activity.\n\nTRIBE v2 predicts an average-subject fMRI response. It does **not** measure the user's own brain activity.

In [ ]:
!python -m pip install -U pip\n!pip install torch>=2.5,<2.7 numpy pandas matplotlib tqdm ffmpeg-python\n!pip install git+https://github.com/facebookresearch/tribev2.git

In [ ]:
from pathlib import Path\nfrom google.colab import files\n\nuploaded = files.upload()\nvideo_path = next(iter(uploaded))\nprint('Uploaded:', video_path)

In [ ]:
from tribev2 import TribeModel\n\nmodel = TribeModel.from_pretrained('facebook/tribev2', cache_folder='./cache')\nevents = model.get_events_dataframe(video_path=video_path)\npreds, segments = model.predict(events=events)\n\nprint('Prediction shape:', preds.shape)\nprint('Number of segments:', len(segments))

In [ ]:
import json\nimport numpy as np\n\nnp.save('predictions.npy', preds)\nwith open('segments.json', 'w') as f:\n    json.dump(segments, f, indent=2, default=str)\n\nsummary = {\n    'shape': list(preds.shape),\n    'mean': float(np.nanmean(preds)),\n    'std': float(np.nanstd(preds)),\n    'min': float(np.nanmin(preds)),\n    'max': float(np.nanmax(preds)),\n}\nprint(json.dumps(summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt\n\nplt.figure(figsize=(12,5))\nplt.imshow(preds[:, ::100].T, aspect='auto', interpolation='nearest')\nplt.xlabel('Prediction timestep')\nplt.ylabel('Cortical vertices (downsampled)')\nplt.title('TRIBE v2 predicted cortical activity')\nplt.colorbar(label='Predicted response')\nplt.tight_layout()

## Optional: cortical surface visualization\nThe official TRIBE v2 package has a plotting extra. Install it with:\n\n`pip install -e ".[plotting]"`\n\nThen use the upstream plotting helpers described in the official repository README.